# 08 存储引擎

## 八股问答

### 0. 概念速览

下面的问答会直接使用这些词，先看一遍避免卡住：

| 概念 | 一句话解释 |
| --- | --- |
| 事务 | 一组 SQL 要么全部成功，要么全部回滚，保证数据一致。 |
| 行锁 | 只锁正在操作的行，其他行仍可读写；并发高，但锁管理开销大。 |
| 表锁 | 锁住整张表，写操作互斥；实现简单，但并发低。 |
| 崩溃恢复 | 数据库异常重启后，用 redo log 恢复还没落盘的已提交修改。 |
| 外键 | 表之间的引用约束，比如订单的 user_id 必须在用户表里存在。 |
| 聚簇索引 | 主键索引的叶子节点直接存整行数据，数据按主键顺序组织。 |
| Buffer Pool | 内存里的数据页和索引页缓存，命中就不用读磁盘。 |
| Change Buffer | 先缓存二级索引的修改，等页被读入时再合并，减少随机写。 |
| Log Buffer | redo log 的内存缓冲，事务提交时刷盘。 |
| redo log | 记录“哪个页被改成什么样”，崩溃时重放恢复。 |
| undo log | 记录旧版本数据，用于回滚和 MVCC。 |
| 页 | InnoDB 最小的磁盘 IO 和内存管理单位，默认 16KB。 |
| 区 | 64 个页组成，默认 1MB。 |
| 表空间 | 数据的逻辑存放单位，用户表对应 .ibd 文件。 |
| 双写缓冲 | 写数据页前先写双写文件，防止写一半崩溃导致页损坏。 |
| Master/IO/Purge 线程 | 分别负责刷脏合并、处理读写 IO、清理旧 undo 和删除标记。 |
| 全文索引 | 针对文本内容的关键词索引，支持全文搜索。 |

### 0.1 存储划分与查询检索

```mermaid
flowchart TD
    subgraph 划分
        TS[表空间] --> SEG[段]
        SEG --> EXT[区：64 个页 = 1MB]
        EXT --> PG[页：16KB]
        PG --> ROW[行：记录]
    end

    subgraph 检索
        Q[SELECT 查询] --> IDX{走哪种索引}
        IDX -- 主键 / 聚簇索引 --> ROOT[B+ 树根页]
        IDX -- 二级索引 --> SEC[二级索引叶子取主键]
        SEC --> ROOT
        ROOT --> MID[中间页]
        MID --> LEAF[叶子页]
        LEAF --> BP{目标页在 Buffer Pool？}
        BP -- 否 --> LOAD[从磁盘读 16KB 页]
        BP -- 是 --> CACHE[直接用内存页]
        LOAD --> DIR[页目录二分查找]
        CACHE --> DIR
        DIR --> REC[定位到行记录]
        REC --> RET[返回结果]
    end
```

### 1. 常见的存储引擎有哪些？

| 引擎 | 特点 | 适用 |
| --- | --- | --- |
| InnoDB | 事务、行锁、崩溃恢复、外键 | 默认，绝大多数业务 |
| MyISAM | 表锁、无事务、压缩和全文索引好 | 只读归档、全文检索旧场景 |
| Memory | 数据全在内存，重启丢失 | 临时表、缓存类小表 |
| Archive | 只支持插入和查询，高压缩 | 日志归档 |
| CSV | 数据就是 CSV 文件 | 与外部工具交换 |
| NDB | 分布式内存引擎 | 高可用分片场景 |

### 2. InnoDB、MyISAM 和 Memory 的区别？

| 维度 | InnoDB | MyISAM | Memory |
| --- | --- | --- | --- |
| 存储限制 | 数据落盘，受文件系统和表空间配置限制 | 数据和索引是独立的 `.MYD`/`.MYI` 文件，受文件系统限制 | 数据只保存在内存，受 `max_heap_table_size`/`tmp_table_size` 和可用内存限制，重启丢失 |
| 事务安全 | 支持 ACID 事务、崩溃恢复、redo/undo | 不支持事务，无崩溃恢复 | 不支持事务，无持久化 |
| 锁机制 | 行锁为主，支持间隙锁、临键锁；DDL 用表锁 | 表级锁，支持并发读，写并发低 | 表级锁，写并发低 |
| B+Tree 索引 | 支持聚簇索引，主键叶子节点直接存储整行 | 支持非聚簇 B+Tree，索引与数据分离 | 支持 BTREE 索引 |
| 哈希索引 | 不提供显式 HASH 索引，只有内部自适应哈希优化 | 不支持 | 支持 HASH 索引，适合等值查询 |
| 全文索引 | 支持（5.6+） | 支持 | 不支持 |
| 空间使用 | 数据和索引统一在表空间/`.ibd`，还有 undo、redo、doublewrite 等开销 | 数据 `.MYD`、索引 `.MYI` 分离，通常更紧凑 | 数据只占内存，磁盘只保存表结构元数据 |
| 内存使用 | 主要用 Buffer Pool 缓存数据页和索引页，可配置 | 索引进 key cache，数据页主要交给操作系统缓存 | 整张表常驻内存，占用较大 |
| 批量插入速度 | 相对较慢，受事务、redo 和索引维护影响；大事务或批量导入可优化 | 通常较快，无事务和行锁开销；但表锁、无崩溃恢复 | 小表最快，数据不落盘；受表锁和内存上限影响 |
| 外键 | 支持 | 不支持 | 不支持 |

结论：默认 InnoDB 是因为它支持事务、行锁和崩溃恢复，适合现代高并发读写业务；
MyISAM 只在极少数纯读、全文检索场景还有存在价值；Memory 适合临时表和小型缓存表，
但数据重启后丢失，不能承担持久化业务。

### 3. InnoDB 底层架构？

从内存、磁盘、后台线程三个角度看：

内存：

- Buffer Pool：数据页和索引页缓存，LRU 管理。
- Change Buffer：缓存二级索引的变更，延迟合并。
- Log Buffer：redo log 的内存缓冲，事务提交时刷盘。

磁盘：

- 表空间：系统表空间（数据字典、双写缓冲）、用户表空间（.ibd 文件）、
  undo 表空间、临时表空间。
- 段-区-页：表空间按段组织，段由区（默认 1MB、64 个页）组成，页默认 16KB。
- redo log：循环写的日志文件组。
- 双写缓冲 Doublewrite：先写双写缓冲再写数据页，防止页断裂（部分写）。

后台线程：

- Master Thread：刷脏、合并 Change Buffer、purge undo 等。
- IO Thread：读写请求处理，包括 redo、change buffer、insert buffer 等。
- Purge Thread：清理已提交事务的 undo 和删除标记。

一条查询的路径：Buffer Pool 命中则直接返回，未命中则从磁盘加载页进 Buffer
Pool，再返回结果。一条写入的路径见 [06-日志与主从复制](06-日志与主从复制.md)。

### 4. 存储引擎选择

在选择存储引擎时，应该根据应用系统的特点选择合适的存储引擎。对于复杂的应用系统，
还可以根据实际情况选择多种存储引擎进行组合。

- InnoDB：是Mysql的默认存储引擎，支持事务、外键。如果应用对事务的完整性有比较高的
  要求，在并发条件下要求数据的一致性，数据操作除了插入和查询之外，还包含很多的更新、
  删除操作，那么InnoDB存储引擎是比较合适的选择。
- MyISAM：如果应用是以读操作和插入操作为主，只有很少的更新和删除操作，并且对事务的
  完整性、并发性要求不是很高，那么选择这个存储引擎是非常合适的。
- MEMORY：将所有数据保存在内存中，访问速度快，通常用于临时表及缓存。MEMORY的缺陷
  就是对表的大小有限制，太大的表无法缓存在内存中，而且无法保障数据的安全性。


## 面试话术

“默认 InnoDB 有三个不可替代的点：事务、行锁、崩溃恢复。它的架构可以按
内存-磁盘-后台线程三块讲：Buffer Pool 和 Log Buffer 负责快，表空间和 redo
负责持久，Master/IO/Purge 线程负责后台刷脏和清理。选引擎时先问业务要不要
事务和并发写，要就 InnoDB；纯归档只读表才考虑 MyISAM 或 Archive。”


## 自查清单

- [ ] 能说出至少五种引擎和各自适用场景
- [ ] 能对比 InnoDB 和 MyISAM 的五个维度
- [ ] 能从内存、磁盘、后台线程三块画出 InnoDB 架构
- [ ] 能画出 InnoDB 逻辑存储结构：表空间 → 段 → 区 → 页 → 行
- [ ] 能解释双写缓冲和 Change Buffer 各解决什么问题
- [ ] 能在终端查看表引擎和 InnoDB 架构参数
